In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

# 1. Load and Balance Data (Crucial for fixing 1.0 Bias)
df = pd.read_csv('CEAS_08.csv')
df = df.dropna(subset=['body', 'label'])

# Map 1 -> 1 (Phishing), 0 -> -1 (Safe)
df['mapped_label'] = df['label'].map({1: 1, 0: -1})
df = df.dropna(subset=['mapped_label'])

# Balance the classes: 50% Phishing, 50% Safe
phish = df[df['mapped_label'] == 1]
safe = df[df['mapped_label'] == -1]
n_samples = min(len(phish), len(safe))

df_balanced = pd.concat([
    phish.sample(n_samples, random_state=42),
    safe.sample(n_samples, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced Dataset Size: {len(df_balanced)} ({n_samples} per class)")

# 2. Vectorization
vectorizer = TfidfVectorizer(max_features=3000, token_pattern=r"(?u)\b\w+\b")
X = vectorizer.fit_transform(df_balanced['body']).toarray()
y = df_balanced['mapped_label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to Tensors and use DataLoader
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

# 3. SVM Model
class LinearSVM(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, 1)
    def forward(self, x): return self.fc(x)

model = LinearSVM(X_train.shape[1])
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam handles sparse TF-IDF better

# 4. Training Loop (Mini-batch Hinge Loss)
print("Training Balanced Model...")
for epoch in range(30): # Fewer epochs needed with Adam + Batches
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        # Hinge Loss + L2 Reg
        loss = torch.mean(torch.clamp(1 - batch_y * outputs, min=0)) + 0.01 * torch.sum(model.fc.weight**2)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Avg Loss: {epoch_loss/len(train_loader):.4f}")

# 5. Evaluation
model.eval()
with torch.no_grad():
    raw_preds = model(X_test_t)
    # Binary decision at 0.0
    preds = torch.where(raw_preds > 0, 1, -1).numpy()

print("\n" + "="*35)
print(f"Accuracy:  {accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, pos_label=1):.4f}")
print(f"Recall:    {recall_score(y_test, preds, pos_label=1):.4f}")
print("="*35)

# Verify Prediction Distribution
unique, counts = np.unique(preds, return_counts=True)
print(f"Final Prediction Counts: {dict(zip(unique, counts))}")




Balanced Dataset Size: 34624 (17312 per class)
Training Balanced Model...
Epoch 0 | Avg Loss: 0.7532
Epoch 10 | Avg Loss: 0.5839
Epoch 20 | Avg Loss: 0.5839

Accuracy:  0.9461
Precision: 0.9319
Recall:    0.9624
Final Prediction Counts: {np.int64(-1): np.int64(3357), np.int64(1): np.int64(3568)}
